1. AI Agents vs Chatbots

2. Tool Calling + Multi-Step Reasoning

3. Text to SQL Architecture

4. Building the Complete Pipeline

5. Mini Project

#AI AGENT -> Large Language Model(LLM)
  -Use tools

  -Take multiple steps

  -Decide which action to take

  -Act on the real world

#COMPONENTS:
  -LLM - Brain/Decision Maker

  -Tools - Functions it can call

  -Memory - Conversation history

  -Reasoning - Multi-step planning

#LLM + SQLite tool = Text-to-SQL Agent

TOOL CALLING:
  
  It means giving thr Ai the ability to eun specific Python fucntions.

  -> get_schema() -->returns database table structure to the AI so it knows column nasme names and types

  -> generate_sql(question) -->Sends user question + schema to groq LLM,returns a SQL query string

  -> execute_sql(query) --> Runs the AI-generated SQL on SQLite and returns result as a dataframe

#Multi-Step Reasoning:The ReAct Pattern

##React = Reasoning + Acting .The agent loops through four steps

Think -->Understand the user's intent

Plan -->What sql is needed ...Which columns,filter,sort order?

Act -->Execute the SQL on the real database.Fetch actual data

Respond -->Formal results.Write the natural language answer

#Technical Architecture:

Stage 1: Schema Injection

              |
    
Stage 2:SQL Generation

              |
Stage 3: Execute+Respond

#Text to SQL Pipeline

   1. load CSV into SQLite

   2.Get Database Schema

   3.Generate SQL with Groq

   4.Execute SQL on SQLite

   5.Natural Language Answer



### **Key Components of an AI Agent:**

| Component | Role | Example |
| --- | --- | --- |
| **LLM** | Brain / Decision Maker | Groq's LLaMA model |
| **Tools** | Actions the agent can take | SQL executor, web search |
| **Memory** | Past conversation context | Chat history |
| **Reasoning** | Multi-step planning | Chain of thought |

---

### **Chatbot vs AI Agent: Side-by-Side**

| Feature | Chatbot | AI Agent |
| --- | --- | --- |
| **Uses tools** | No | Yes |
| **Multi-step reasoning** | No | Yes |
| **Can query databases** | No | Yes |





In [ ]:
!pip install groq -q
print("Libraries installed successfully")


Libraries installed successfully


In [ ]:
import sqlite3
import pandas as pd
import os
from groq import Groq
import re
print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
import os
os.environ["GROQ_API_KEY"]="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"
client=Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL="llama-3.1-8b-instant"

print(f"Groq client initialized successfully")
print(f"Using model:{MODEL}")

Groq client initialized successfully
Using model:llama-3.1-8b-instant


In [ ]:
import io

csv_data = """student_id,name,age,gender,subject,marks,attendance,grade
1,Aarav Sharma,20,Male,Mathematics,88,92,A
2,Priya Patel,21,Female,Science,76,85,B
3,Rohan Mehta,20,Male,Programming,95,98,A+
4,Sneha Iyer,22,Female,Mathematics,62,78,C
5,Arjun Nair,21,Male,Programming,91,94,A+
6,Divya Krishnan,20,Female,Science,83,88,A
7,Karan Singh,22,Male,Mathematics,74,81,B
8,Ananya Gupta,21,Female,Programming,89,96,A
9,Vikram Reddy,20,Male,Science,70,79,B
10,Pooja Sharma,22,Female,Mathematics,55,72,D
11,Aditya Kumar,21,Male,Programming,97,99,A+
12,Meera Nambiar,20,Female,Science,81,87,A
13,Rahul Desai,22,Male,Mathematics,68,80,C
14,Kavitha Rajan,21,Female,Programming,86,93,A
15,Nikhil Verma,20,Male,Science,77,84,B
16,Swathi Pillai,22,Female,Mathematics,90,95,A+
17,Manish Joshi,21,Male,Programming,73,82,B
18,Lavanya Menon,20,Female,Science,66,76,C
19,Suresh Babu,22,Male,Mathematics,82,89,A
20,Anjali Singh,21,Female,Programming,94,97,A+
21,Deepak Nair,20,Male,Science,79,86,B
22,Rekha Sharma,22,Female,Mathematics,58,73,D
23,Sanjay Patel,21,Male,Programming,88,91,A
24,Usha Iyer,20,Female,Science,84,90,A
25,Vijay Kumar,22,Male,Mathematics,71,83,B
26,Nandita Rao,21,Female,Programming,92,96,A+
27,Ashok Reddy,20,Male,Science,65,77,C
28,Sunita Gupta,22,Female,Mathematics,87,93,A
29,Ravi Krishnan,21,Male,Programming,78,88,B
30,Bhavna Mehta,20,Female,Science,93,98,A+"""
df=pd.read_csv(io.StringIO(csv_data))
print(f"Dataset loaded: {len(df)} rows,{len(df.columns)} columns")
print("\nFirst 5 rows")
df.head()





Dataset loaded: 30 rows,8 columns

First 5 rows


,student_id,name,age,gender,subject,marks,attendance,grade
0,1,Aarav Sharma,20,Male,Mathematics,88,92,A
1,2,Priya Patel,21,Female,Science,76,85,B
2,3,Rohan Mehta,20,Male,Programming,95,98,A+
3,4,Sneha Iyer,22,Female,Mathematics,62,78,C
4,5,Arjun Nair,21,Male,Programming,91,94,A+


In [ ]:
conn=sqlite3.connect("college.db")
df.to_sql("students",conn,if_exists="replace",index=False)
print("Database created successfully : college.db")
print("Table 'students' created with 30 student records")
test_df=pd.read_sql_query("SELECT COUNT(*) as total_rows FROM students",conn)

print(f"\nVerification: {test_df['total_rows'][0]} rows in database")


Database created successfully : college.db
Table 'students' created with 30 student records

Verification: 30 rows in database


In [ ]:
def get_schema(conn, table_name="students"):
  cursor=conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")

  columns=cursor.fetchall()

  schema_lines=[f"Table : {table_name}"]
  schema_lines.append("Columns : ")

  for col in columns :
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")

  sample_rows=cursor.fetchall()
  schema_lines.append("\nSample rows (first 3) : ")

  for row in sample_rows :
    schema_lines.append(f"  - {row}")

  return "\n".join(schema_lines)

schema=get_schema(conn)
print(schema)

Table : students
Columns : 
  - student_id (INTEGER)
  - name (TEXT)
  - age (INTEGER)
  - gender (TEXT)
  - subject (TEXT)
  - marks (INTEGER)
  - attendance (INTEGER)
  - grade (TEXT)

Sample rows (first 3) : 
  - (1, 'Aarav Sharma', 20, 'Male', 'Mathematics', 88, 92, 'A')
  - (2, 'Priya Patel', 21, 'Female', 'Science', 76, 85, 'B')
  - (3, 'Rohan Mehta', 20, 'Male', 'Programming', 95, 98, 'A+')


In [ ]:
def generate_sql(user_question, schema_text, client, model):

  system_prompt=f"""You are an expert SQL assistant. You are connected to a SQLite database with the following structure:

  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or  text -  only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is : students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses( example: WHERE subject = 'Programming')
  7. If the user asks for top N , use ORDER BY marks DESC LIMIT N.

  """
  response=client.chat.completions.create(
      model=model,

      messages=[
          {"role":"system", "content":system_prompt},
          {"role":"user", "content":user_question}
      ],
      temperature=0.0
  )

  sql_query=response.choices[0].message.content.strip()

  return sql_query

question="Show me all female students"
print(f"Question : {question}")
print("\nGenerating SQL...")

sql=generate_sql(question, schema, client, MODEL)
print(f"\nGenerated SQL : \n{sql}")

Question : Show me all female students

Generating SQL...

Generated SQL : 
SELECT * FROM students WHERE gender = 'Female'


In [ ]:
def execute_sql(sql_query, conn):
  clean_sql = sql_query.strip()

  clean_sql = re.sub(r'```sql\s*', '' , clean_sql)

  clean_sql = clean_sql.strip()


  try:

    result_df = pd.read_sql_query(clean_sql,conn)

    return result_df , None

  except Exception as e:
    return None , str(e)


print(f"Executing SQL: {sql}")
result, error = execute_sql(sql, conn)

if error:
  print(f"Error: {error}")
else:
  print(f"\nQuery returned {len(result)} rows")
  print(result)

Executing SQL: SELECT * FROM students WHERE gender = 'Female'

Query returned 15 rows
    student_id            name  age  gender      subject  marks  attendance  \
0            2     Priya Patel   21  Female      Science     76          85   
1            4      Sneha Iyer   22  Female  Mathematics     62          78   
2            6  Divya Krishnan   20  Female      Science     83          88   
3            8    Ananya Gupta   21  Female  Programming     89          96   
4           10    Pooja Sharma   22  Female  Mathematics     55          72   
5           12   Meera Nambiar   20  Female      Science     81          87   
6           14   Kavitha Rajan   21  Female  Programming     86          93   
7           16   Swathi Pillai   22  Female  Mathematics     90          95   
8           18   Lavanya Menon   20  Female      Science     66          76   
9           20    Anjali Singh   21  Female  Programming     94          97   
10          22    Rekha Sharma   22  Female  

Rules you must follow:
1. Generate ONLY a valid SQLite SQL query.
2. Do not include any explanation or  text -  only the SQL query.
3. Do not use markdown code blocks. Return the raw SQL only.
4. The table name is : students
5. Only use column names that exist in the schema above.
6. Use single quotes for string values in WHERE clauses( example: WHERE subject = )
7. If the user asks for top N , use ORDER BY marks DESC LIMIT N.

"""

In [ ]:
def text_to_sql_agent(user_question, conn, client , model , verbose=True):
  print("=" * 60)
  print(f"User Question: {user_question}")
  print("=" * 60)

  if verbose:
    print("\n[STEP 1] Reading database schema...")

  schema_text = get_schema(conn)

  if verbose:
    print("Schema loaded successfully")

  if verbose:
    print("\n[STEP 2] Generating SQL query with Groq LLM...")

  generated_sql = generate_sql(user_question, schema_text, client, model)

  if verbose:
    print(f"Generated sql:\n {generated_sql}")

  if verbose:
    print("\n[STEP3] Executing SQL on the database...")

  result_df, error=execute_sql(generated_sql,conn)

  if error:
    print(f"SQL Execution error : {error}")
    return None, generated_sql

  if verbose:
    print(f"\n[STEP 4] Query returned {len(result_df)} row(s)")
    print("\nRESULTS:")
    print("-" *40)
    print(result_df.to_string(index=False))
    print("=" * 60)

    return result_df , generated_sql

result, sql_used = text_to_sql_agent(
     "Show top 5 students  in Programming",
     conn,client,MODEL
 )

User Question: Show top 5 students  in Programming

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
        name
Aditya Kumar
 Rohan Mehta
Anjali Singh
 Nandita Rao
  Arjun Nair


In [ ]:
result1, _ = text_to_sql_agent(
    "Show me all students who study Mathematics",
    conn, client, MODEL
)

User Question: Show me all students who study Mathematics

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT * FROM students WHERE subject = 'Mathematics'

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 10 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
          1  Aarav Sharma   20   Male Mathematics     88          92     A
          4    Sneha Iyer   22 Female Mathematics     62          78     C
          7   Karan Singh   22   Male Mathematics     74          81     B
         10  Pooja Sharma   22 Female Mathematics     55          72     D
         13   Rahul Desai   22   Male Mathematics     68          80     C
         16 Swathi Pillai   22 Female Mathematics     90          95    A+
         19   Suresh Babu   22   Male Mathematics     82          89     A
         22  Rekha Sharma   22 Fe

In [ ]:
result2, _ = text_to_sql_agent(
    "What is the average marks for each subject?",
    conn, client, MODEL
)

User Question: What is the average marks for each subject?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT subject, AVG(marks) FROM students GROUP BY subject

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 3 row(s)

RESULTS:
----------------------------------------
    subject  AVG(marks)
Mathematics        73.5
Programming        88.3
    Science        77.4


In [ ]:
result3, _ = text_to_sql_agent(
    "Show students who scored more than 90 marks",
    conn, client, MODEL
)

User Question: Show students who scored more than 90 marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT * FROM students WHERE marks > 90

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 6 row(s)

RESULTS:
----------------------------------------
 student_id         name  age gender     subject  marks  attendance grade
          3  Rohan Mehta   20   Male Programming     95          98    A+
          5   Arjun Nair   21   Male Programming     91          94    A+
         11 Aditya Kumar   21   Male Programming     97          99    A+
         20 Anjali Singh   21 Female Programming     94          97    A+
         26  Nandita Rao   21 Female Programming     92          96    A+
         30 Bhavna Mehta   20 Female     Science     93          98    A+


In [ ]:
result4, _ = text_to_sql_agent(
    "How many male and female students are there?",
    conn, client, MODEL
)

User Question: How many male and female students are there?

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT COUNT(CASE WHEN gender = 'Male' THEN 1 END) AS male_count, 
       COUNT(CASE WHEN gender = 'Female' THEN 1 END) AS female_count 
FROM students

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 1 row(s)

RESULTS:
----------------------------------------
 male_count  female_count
         15            15


In [ ]:
result5, _ = text_to_sql_agent(
    "Show female students who scored above 85 in Science or Programming, ordered by marks",
    conn, client, MODEL
)

User Question: Show female students who scored above 85 in Science or Programming, ordered by marks

[STEP 1] Reading database schema...
Schema loaded successfully

[STEP 2] Generating SQL query with Groq LLM...
Generated sql:
 SELECT * FROM students WHERE gender = 'Female' AND subject IN ('Science', 'Programming') AND marks > 85 ORDER BY marks DESC

[STEP3] Executing SQL on the database...

[STEP 4] Query returned 5 row(s)

RESULTS:
----------------------------------------
 student_id          name  age gender     subject  marks  attendance grade
         20  Anjali Singh   21 Female Programming     94          97    A+
         30  Bhavna Mehta   20 Female     Science     93          98    A+
         26   Nandita Rao   21 Female Programming     92          96    A+
          8  Ananya Gupta   21 Female Programming     89          96     A
         14 Kavitha Rajan   21 Female Programming     86          93     A


YOU TYPE: "Show top 5 students in Programming"

           |
           v
    [AGENT reads the database schema]
    Table: students
    Columns: student_id, name, age, subject, marks ...
           |
           v
    [AGENT sends to Groq LLM]
    System: You are an SQL expert. Here is the schema: ...
    User: Show top 5 students in Programming
           |
           v
    [LLM generates SQL]
    SELECT name, marks FROM students
    WHERE subject = 'Programming'
    ORDER BY marks DESC
    LIMIT 5
           |
           v
    [AGENT executes SQL on SQLite]
    Runs the query on college.db
           |
           v
    [AGENT returns results]
    Aditya Kumar   97
    Anjali Singh   94
    ...
Why This Is Powerful
Non-technical users can ask complex database questions without knowing SQL. Business analysts, managers, teachers, and students can all query data using plain language. This is exactly how products like Amazon QuickSight Q, Google Looker, and Tableau AI work.

In [ ]:

def generate_answer(user_question, query_results_df, client, model):


    if query_results_df is None or len(query_results_df) == 0:
        return "No results were found for your query."

    results_text = query_results_df.to_string(index=False)


    prompt = f"""The user asked: '{user_question}'

The database returned these results:
{results_text}

Please write a clear, friendly, 2-3 sentence answer to the user's question based on these results.
Be specific. Mention actual names and numbers from the data.
Do not add information not present in the results."""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3

    )

    return response.choices[0].message.content.strip()



def smart_text_to_sql_agent(user_question, conn, client, model):
    """
    Enhanced agent that returns both a data table AND a natural language answer.
    """
    print("=" * 60)
    print(f"Question: {user_question}")
    print("=" * 60)

    schema_text = get_schema(conn)

    print("Generating SQL...")
    generated_sql = generate_sql(user_question, schema_text, client, model)
    print(f"SQL: {generated_sql}")

    result_df, error = execute_sql(generated_sql, conn)

    if error:
        print(f"Error executing SQL: {error}")
        return

    print(f"\nData ({len(result_df)} rows returned):")
    display(result_df)

    print("\nGenerating natural language answer...")
    answer = generate_answer(user_question, result_df, client, model)

    print("\nAnswer:")
    print(answer)
    print("=" * 60)


smart_text_to_sql_agent(
    "Who are the top 5 students in Programming?",
    conn, client, MODEL
)

Question: Who are the top 5 students in Programming?
Generating SQL...
SQL: SELECT name FROM students WHERE subject = 'Programming' ORDER BY marks DESC LIMIT 5

Data (5 rows returned):


,name
0,Aditya Kumar
1,Rohan Mehta
2,Anjali Singh
3,Nandita Rao
4,Arjun Nair



Generating natural language answer...

Answer:
Based on the results, it seems that the top 5 students in Programming are Aditya Kumar, Rohan Mehta, Anjali Singh, Nandita Rao, and Arjun Nair. These students are currently at the top of the list, but please note that this information may not reflect their overall performance or ranking in the course.


In [ ]:
smart_text_to_sql_agent(
    "Which subject has the highest average attendance?",
    conn, client, MODEL
)

Question: Which subject has the highest average attendance?
Generating SQL...
SQL: SELECT subject FROM students GROUP BY subject ORDER BY AVG(attendance) DESC LIMIT 1

Data (1 rows returned):


,subject
0,Programming



Generating natural language answer...

Answer:
Based on the results, it appears that the subject with the highest average attendance is actually "Programming" with a single entry.


#Section 14: Practice Questions:

##Beginner Questions:

1.What is the difference between a chatbot and an AI Agent? Give one example of each.

2.What is schema injection? Why does the AI need to know the schema before generating SQL?

3.Why do we set temperature=0.0 when generating SQL queries?

##Intermediate Questions:

1.In the generate_sql() function, what is the role of the system message vs the user message?

2.What does PRAGMA table_info(students) return? What information does it give us?

3.If the LLM generates incorrect SQL, what strategies can you use to improve it?

##Coding Questions:

1.Modify generate_sql() to also handle aggregate functions like COUNT and SUM.
Test it with the question: "How many students have an attendance above 90?"

2.Add a new function get_distinct_values(column_name, conn) that returns all unique values in a column. Use this to show the AI what values exist in the subject column.

3.Build a simple loop that lets the user type questions and see results until they type 'exit'.

In [ ]:
smart_text_to_sql_agent(
    "How many students have attendance above 90?",
    conn, client, MODEL
)

Question: How many students have attendance above 90?
Generating SQL...
SQL: SELECT COUNT(*) FROM students WHERE attendance > 90

Data (1 rows returned):


,COUNT(*)
0,12



Generating natural language answer...

Answer:
Based on the results, it appears that 12 students have attendance above 90%. We don't have any additional information about these students, but the count suggests that a significant number of students are meeting this attendance standard.


#Beginner Questions
1.
Chatbot

Responds to user queries through conversation.

Mostly reactive.

Usually answers questions only.

AI Agent

Can reason, make decisions, and perform actions to achieve goals.

Proactive and goal-oriented.

Can use tools, APIs, databases, and perform tasks.

Example Chatbot: ChatGPT customer support bot.

Example AI Agent: An AI assistant that books meetings, sends emails, and updates calendars automatically.

2.

Schema injection is the process of providing the database schema (table names, column names, data types, etc.) to the LLM before asking it to generate SQL.


The AI needs the schema because:


It must know which tables exist.

It must know available columns.

It prevents invalid SQL queries.

It improves query accuracy.


Example: If the database has a column attendance_percentage, the AI should not generate a query using a non-existent column like attendance.


3.
SQL generation requires accuracy and consistency.

Setting:

temperature = 0.0

means:

Deterministic outputs.

Less randomness.

More reliable SQL generation.

Same question → Same SQL query.

This reduces the chance of generating invalid or creative SQL statements.

#Intermediate Questions
1.
System Message

Provides instructions and rules to the AI.

Example:

You are an expert SQL generator.

Generate only SQLite queries.

It controls the AI's behavior.

User Message

Contains the actual question.

Example:

Show all students with attendance above 90.

The AI uses both messages to generate the SQL query.

2.

It returns metadata about the table structure.

Example:

PRAGMA table_info(students);

Output:

cid	name	type	notnull	dflt_value

0	id	INTEGER	0	NULL	1

1	name	TEXT	0	NULL	0

2	attendance	REAL	0	NULL	0

Information provided:

Column names

Data types

Primary key information

Default values

NULL constraints

This helps the AI understand the database structure.

3.
Provide schema information.

Use better prompts.

Set temperature to 0.0.

Give SQL examples (few-shot prompting).

Validate SQL before execution.

Show column descriptions.

Provide sample rows from the table.

Use error feedback and regenerate SQL.

In [ ]:

#7
def get_distinct_values(column_name, table_name, conn):
    """
    Returns all unique values in a specific column.
    Useful for understanding what data exists in categorical columns.

    Parameters:
        column_name: The column to get unique values from
        table_name: The table to query
        conn: Database connection
    """
    query = f"SELECT DISTINCT {column_name} FROM {table_name} ORDER BY {column_name}"


    result = pd.read_sql_query(query, conn)
    return result[column_name].tolist()


# Test it
subjects = get_distinct_values("subject", "students", conn)
print(f"Available subjects in database: {subjects}")

grades = get_distinct_values("grade", "students", conn)
print(f"Available grades in database: {grades}")

Available subjects in database: ['Mathematics', 'Programming', 'Science']
Available grades in database: ['A', 'A+', 'B', 'C', 'D']


In [ ]:
#8 and 9

def run_interactive_agent(conn, client, model):
    """
    Runs an interactive loop that accepts user questions until 'exit' is typed.
    This simulates a real chatbot interface for our Text-to-SQL system.
    """
    print("Text-to-SQL Agent Ready")
    print("Type your question about the student database.")
    print("Type 'exit' to quit.")
    print("-" * 40)

    while True:

        question = input("\nYour question: ").strip()


        if question.lower() == "exit":

            print("Exiting agent. Goodbye.")
            break

        if not question:

            print("Please type a question.")
            continue

        smart_text_to_sql_agent(question, conn, client, model)




print("Interactive agent function is ready.")
print("Uncomment run_interactive_agent() to launch it.")

Interactive agent function is ready.
Uncomment run_interactive_agent() to launch it.


## Mini Project: Natural Language SQL Dashboard

---

### Project Goal

Build a complete Natural Language to SQL system that:
1. Accepts a question from the user
2. Generates the SQL query
3. Executes it on the students database
4. Displays the results as a table
5. Shows a bar chart of the results (if numeric data is present)
6. Provides a natural language answer

**Time:** 35–50 minutes

**Step-by-step guidance below.**